In [ ]:
import json
import sys
from collections import defaultdict
from datetime import datetime

import polars as pl
from Bio import SeqIO
from ete4 import Tree

sys.path.insert(0, "..")
from project import (
    check_run_settings,
    excluded_pairs,
    get_all,
    log,
    update_run_json,
)

In [ ]:
ALL = get_all(["plants_plt"])
# ALL = {"plants_plt": {"solanaceae": get_info("plants_plt", "solanaceae")}}
OVERWRITE = False  # True allows changing the settings of an existing guide_trees run

DUP_BRANCH = 1e-4  # branch length for sister tips added when one genome has several IGRs in a gene pair

In [ ]:
def scale_tree(tree: Tree, factor: float) -> None:
    """Multiply every branch length in place."""
    if factor == 1:
        return
    for node in tree.traverse():
        if node.dist:
            node.dist *= factor


def gene_pair_tree(subg_tree: Tree, ids_by_an: dict[str, list[str]]) -> Tree:
    """Subgroup tree pruned to the ANs present, tips relabelled to IGR IDs."""
    tree = subg_tree.copy()
    tree.prune(list(ids_by_an), preserve_branch_length=True)

    for leaf in list(tree.leaves()):
        ids = ids_by_an[leaf.name]
        if len(ids) == 1:
            leaf.name = ids[0]
            continue
        # several IGRs from one genome (e.g. IR copies): turn the tip into a node with one child per IGR
        leaf.name = ""
        for igr_id in ids:
            leaf.add_child(name=igr_id, dist=DUP_BRANCH)

    tree.resolve_polytomy()  # 3+ copies from one genome would be a polytomy
    return tree

In [ ]:
for g, subs in ALL.items():
    for subg, info in subs.items():
        p, cfg = info["paths"], info["config"]
        iqtree_run, guide_run = info["runs"]["iqtree"], info["runs"]["guide_trees"]
        log(
            f"{g}/{subg} | guide_trees run: {guide_run} | species tree: iqtree/{iqtree_run}"
        )

        if "guide_tree_scale" not in cfg:
            raise KeyError(
                f"guide_tree_scale missing from {p['yaml'].name} (use 1 for the first pass)"
            )
        scale = float(cfg["guide_tree_scale"])
        scale_source = cfg.get("guide_tree_scale_source", "")

        if not p["gene_pairs_filtered"].exists():
            log(
                f"  no {p['gene_pairs_filtered'].name}, run 02_gene_pairs first; skipping"
            )
            continue

        iqtree_dir = p["species_tree"] / "iqtree" / iqtree_run
        src_tree_file = iqtree_dir / "tree.nwk"
        iqtree_json = iqtree_dir / "run.json"
        iqtree_info = (
            json.loads(iqtree_json.read_text()) if iqtree_json.exists() else {}
        )
        if iqtree_info.get("status") != "complete" or not src_tree_file.exists():
            raise RuntimeError(
                f"iqtree/{iqtree_run} is not complete (status: {iqtree_info.get('status')}); run 04_iq_tree"
            )

        run_dir = p["guide_trees"] / guide_run
        run_json = run_dir / "run.json"
        settings = {
            "iqtree_run_name": iqtree_run,
            "scale": scale,
            "scale_source": scale_source,
            "dup_branch": DUP_BRANCH,
            "excluded": sorted(excluded_pairs(info)),
        }
        check_run_settings(run_json, settings, OVERWRITE)

        # --- subgroup tree: AN tips, rooted, scaled
        subg_tree = Tree(src_tree_file.read_text(), parser=5)
        scale_tree(subg_tree, scale)
        tree_ans = {leaf.name for leaf in subg_tree.leaves()}
        depth = max(
            subg_tree.get_distance(subg_tree, leaf) for leaf in subg_tree.leaves()
        )
        log(
            f"  species tree: {len(tree_ans)} tips | max root-to-tip {depth:.4f} (scale {scale})"
        )

        # --- IGR ID -> AN
        atlas = pl.read_csv(p["atlas"], separator="\t")
        id_to_an = dict(
            atlas.select(
                pl.col("ID").cast(pl.String), pl.col("AN").cast(pl.String)
            ).iter_rows()
        )

        # --- pass 1: check every gene pair; collect all problems before writing anything
        gp_df = pl.read_csv(p["gene_pairs_filtered"], separator="\t")
        exclude = excluded_pairs(info, gp_df["gene_pair"])
        gp_df = gp_df.filter(~pl.col("gene_pair").is_in(list(exclude)))
        if exclude:
            log(f"  excluded (exclude_pairs): {sorted(exclude)}")
        plans, problems = {}, []
        unplaced_ans: dict[str, int] = defaultdict(
            int
        )  # AN not in tree -> number of gene pairs affected
        unknown_ids = 0
        for gp in gp_df["gene_pair"]:
            fasta = p["igr_by_pair"] / f"gp_{gp}.fasta"
            if not fasta.exists():
                problems.append(f"{gp}: missing {fasta.name}")
                continue

            ids_by_an: dict[str, list[str]] = defaultdict(list)
            unplaced = []
            for rec in SeqIO.parse(fasta, "fasta"):
                an = id_to_an.get(rec.id)
                if an in tree_ans:
                    ids_by_an[an].append(rec.id)
                else:
                    unplaced.append(rec.id)
                    if an is None:
                        unknown_ids += 1
                    else:
                        unplaced_ans[an] += 1

            if unplaced:
                # PRANK fails if a FASTA sequence has no tip in the tree
                problems.append(
                    f"{gp}: {len(unplaced)} IGRs without a tip ({unplaced[:3]}...)"
                )
                continue
            if len(ids_by_an) < 2:
                problems.append(f"{gp}: fewer than 2 genomes")
                continue
            plans[gp] = ids_by_an

        if problems:
            print(f"  {len(problems)} of {gp_df.height} gene pairs have problems:")
            for prob in problems:
                print(f"    {prob}")
            if unplaced_ans:
                print(
                    "  ANs not in the tree, with the number of gene pairs they appear in:"
                )
                for an, n in sorted(unplaced_ans.items(), key=lambda x: -x[1]):
                    print(f"    {an}: {n}")
            if unknown_ids:
                print(f"  IGR IDs not found in {p['atlas'].name}: {unknown_ids}")
            raise RuntimeError(
                f"{g}/{subg}: gene pairs do not match the tree; nothing was written."
            )

        # --- pass 2: one tree per gene pair; folder rebuilt so dropped gene pairs don't linger
        run_dir.mkdir(parents=True, exist_ok=True)
        for old in run_dir.glob("gp_*.nwk"):
            old.unlink()
        n_dup_genomes = 0
        for gp, ids_by_an in plans.items():
            n_dup_genomes += sum(len(ids) > 1 for ids in ids_by_an.values())
            gene_pair_tree(subg_tree, ids_by_an).write(
                outfile=str(run_dir / f"gp_{gp}.nwk"), parser=5
            )

        update_run_json(
            run_json,
            run_name=guide_run,
            group=g,
            subgroup=subg,
            date=f"{datetime.now():%Y-%m-%d %H:%M:%S}",
            **settings,
            source_tree=str(src_tree_file.relative_to(p["dir"])),
            mafft_run_name=iqtree_info.get("mafft_run_name"),
            cds_run_name=iqtree_info.get("cds_run_name"),
            gene_pairs_file=str(p["gene_pairs_filtered"].relative_to(p["dir"])),
            igr_fastas=str(p["igr_by_pair"].relative_to(p["dir"])),
            n_tree_tips=len(tree_ans),
            max_root_to_tip=depth,
            n_gene_pair_trees=len(plans),
        )
        log(
            f"  {len(plans)}/{gp_df.height} guide trees -> {run_dir.relative_to(p['dir'])}/"
            f" | genomes with several IGRs (sister tips): {n_dup_genomes}"
        )

log("Done")